# Benchmark CUDA en Google Colab

Mi PC no tiene gráfica NVIDIA, así que la parte de CUDA la ejecuto aquí.

1. `Entorno de ejecución > Cambiar tipo de entorno de ejecución > GPU` (vale la T4 gratuita).
2. `Entorno de ejecución > Ejecutar todas`.

Al final se descargan el CSV con los tiempos y las gráficas, que van a `resultados/` y `docs/img/` en el repositorio.

In [ ]:
!nvidia-smi --query-gpu=name,compute_cap,memory.total --format=csv

In [ ]:
!git clone --depth 1 https://github.com/alejandrorevillaEng/gradientes-paralelos.git
%cd gradientes-paralelos

In [ ]:
# 75 = Turing (T4). Si Colab te da otra GPU, cambia la arquitectura (80 = A100, 89 = L4).
!cmake -S . -B build -DCMAKE_BUILD_TYPE=Release
!cmake --build build -j

In [ ]:
!cd build && ctest --output-on-failure

In [ ]:
!./build/gradbench_cuda --reps 5 --csv resultados/gpu_colab.csv

In [ ]:
!./build/gradbench --experimento escalado --reps 3 --csv resultados/cpu_colab.csv

In [ ]:
import subprocess
gpu = subprocess.run(["nvidia-smi", "--query-gpu=name", "--format=csv,noheader"],
                     capture_output=True, text=True).stdout.strip().replace("NVIDIA ", "")
!python bench/graficas.py --gpu resultados/gpu_colab.csv --nombre-gpu "{gpu}" --salida docs/img

In [ ]:
from IPython.display import Image, display
for nombre in ["gpu_bloques.png", "gpu_desglose.png"]:
    display(Image(f"docs/img/{nombre}"))

In [ ]:
from google.colab import files
for ruta in ["resultados/gpu_colab.csv", "resultados/cpu_colab.csv",
             "docs/img/gpu_bloques.png", "docs/img/gpu_desglose.png"]:
    files.download(ruta)